# 📊 Smart E-Commerce Profit Optimizer: Exploratory Data Analysis & Preprocessing
**Author:** Prathamesh Mistari  
**Degree / Specialization:** BCA (4-Year Hons / Sem 7) | SPU Guidelines  
**Repository:** [pratham-mistari/smart-profit-optimizer](https://github.com/pratham-mistari/smart-profit-optimizer)

---
### 🎯 Business Objective
Retail e-commerce enterprises frequently implement aggressive discounting strategies to accelerate top-line Gross Merchandise Value (GMV). However, unconstrained discounting often erodes net operating margins, leading to **silent profit leakage**—where high-revenue categories incur substantial net negative returns.

This notebook conducts end-to-end data auditing, missing value imputation, outlier diagnosis, and exploratory margin analysis to identify key leakage drivers.

In [ ]:
# 1. Environment Setup & Library Ingestion
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Aesthetic plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = 'Segoe UI'
print('Libraries successfully imported!')

## 2. Ingestion of Raw Transactional Data

In [ ]:
raw_path = os.path.join('..', 'data', 'raw', 'superstore_raw.csv')
df = pd.read_csv(raw_path)
print(f'Raw Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## 3. Data Integrity & Missing Value Profiling

In [ ]:
print('--- Dataset Info ---')
df.info()

print('\n--- Missing Value Count per Column ---')
missing_summary = df.isnull().sum()
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else 'Zero missing values detected. High dataset completeness.')

## 4. Feature Engineering: Financial Margins & Flags

In [ ]:
# Date conversions
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df['Fulfillment_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Profit Margin % and Loss Identification
df['Profit_Margin_%'] = np.where(df['Sales'] > 0, (df['Profit'] / df['Sales']) * 100, 0).round(2)
df['Is_Loss_Making'] = df['Profit'] < 0
df['COGS'] = df['Sales'] - df['Profit']

# Categorize discount tiers
bins = [-0.01, 0.0, 0.10, 0.20, 0.40, 1.0]
labels = ['0% None', '1%-10% Low', '11%-20% Moderate', '21%-40% High', '>40% Extreme']
df['Discount_Tier'] = pd.cut(df['Discount'], bins=bins, labels=labels)

print('Feature engineering complete. Sample computed rows:')
df[['Order ID', 'Sales', 'Discount', 'Profit', 'Profit_Margin_%', 'Is_Loss_Making', 'Discount_Tier']].head()

## 5. Exploratory Data Analysis (EDA)
### 5.1 The Discount Leakage Cliff

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(
    data=df.sample(min(1000, len(df)), random_state=42),
    x='Discount',
    y='Profit_Margin_%',
    hue='Category',
    alpha=0.7,
    palette='tab10',
    ax=ax
)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Break-Even (0% Margin)')
ax.set_title('Discount vs. Profit Margin %: Diagnosing Margin Erosion', fontsize=14, fontweight='bold')
ax.set_xlabel('Discount Rate Applied')
ax.set_ylabel('Profit Margin (%)')
ax.legend()
plt.tight_layout()
plt.show()

### 5.2 Category & Sub-Category Profitability Breakdown

In [ ]:
subcat_summary = df.groupby(['Category', 'Sub-Category']).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean'),
    Loss_Order_Count=('Is_Loss_Making', 'sum')
).reset_index()

subcat_summary['Margin_%'] = (subcat_summary['Total_Profit'] / subcat_summary['Total_Sales'] * 100).round(1)
subcat_summary = subcat_summary.sort_values(by='Total_Profit', ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#10B981' if p > 0 else '#EF4444' for p in subcat_summary['Total_Profit']]
sns.barplot(
    data=subcat_summary,
    x='Total_Profit',
    y='Sub-Category',
    palette=colors,
    ax=ax
)
ax.set_title('Cumulative Operating Profit by Sub-Category (Green=Profit, Red=Net Loss)', fontsize=14, fontweight='bold')
ax.set_xlabel('Cumulative Operating Profit ($)')
ax.set_ylabel('Sub-Category')
plt.tight_layout()
plt.show()

## 6. Export Cleaned Dataset for Streamlit Dashboard

In [ ]:
clean_path = os.path.join('..', 'data', 'processed', 'superstore_clean.csv')
os.makedirs(os.path.dirname(clean_path), exist_ok=True)
df.to_csv(clean_path, index=False)
print(f'✅ Successfully exported cleaned dataset to {clean_path}')
print(f'Final Clean Dataset: {df.shape[0]} rows, {df.shape[1]} columns ready for Streamlit engine.')